# Agent Tools Exploration — Input → Tool → Output

The simulation step (`src/runs/run.py`, i.e. `run_simulation.ipynb` — step 7 of the pipeline) gives the LLM a
fixed set of function-calling **tools**. Definitions live in `src/tools.py` (Pydantic schemas), the code that
actually executes when a tool is "called" lives in `src/tool_execs.py`, and registration (which schemas are
exposed, and the name→function map) happens inline in `src/runs/run.py`.

**These tools are only used during the simulation step.** They are not used during dataset building (step 3) or
FHIR seeding (step 4) — those steps just produce the artifacts (the diagnosis dataset, the running FHIR server,
the Qdrant procedure collection) that the tools read from / write to at simulation time.

| # | Tool (LLM-facing) | Exec function | Talks to | Baseline pipeline? |
|---|---|---|---|---|
| 1 | `PhysicalExamination` | `get_physical_exam_results` | dataset (extracted PE text) + FHIR | yes |
| 2 | `LabRequestList` | `get_blood_value_results` | dataset (`lab_events`) + FHIR | yes |
| 3 | `UrineRequestList` | `get_urine_value_results` | dataset (`lab_events`, fluid=Urine) + FHIR | yes |
| 4 | `MicrobiologyRequestList` | `get_microbiology_results` | dataset (`microbiology`) + FHIR | yes |
| 5 | `RadiologyRequestFHIR` | `get_radiology_results` | dataset (`radiology`) + FHIR | yes |
| 6 | `MedicationRequestList` | `get_medication_results` | simulated confirmation (no ground truth) + FHIR | yes |
| 7 | `ProcedureSearch` | `get_procedure_search_results` | Qdrant (`localhost:6333`) semantic search + FHIR | yes |
| 8 | `ProcedureRequestFHIR` | `get_procedure_request_results` | local ICD catalog / Qdrant fallback + FHIR | yes |
| 9 | `Finish` | `finish` | nothing (pure local, ends the case) | yes |
| 10 | `Plan` | `generate_routine` | OpenAI `o1` directly (no FHIR/Qdrant/dataset) | yes |
| 11 | `PatientHistory` | `patient_history` | local JSON (`resources/pancreatic_cancer_info.json`) | only for `ds_name == "pancreatic_cancer"` |

Bonus tools used only in the bias/optional-admission run variants (`run_with_sex_bias.py`, `run_optional_admission.py`),
not the baseline: `VitalSigns` (reads `triage`), `CloseCase` (replaces `Finish`, adds an explicit admit/discharge decision).

Every "talks to FHIR" tool follows the same pattern (`tool_execs.request_fetch_and_poll`): build a FHIR request
resource → POST it to the local HAPI server → fetch the matching ground truth from the pre-built diagnosis
dataset → build a FHIR result resource with a human-readable `note` → POST that too → return the note text as
the string the LLM sees.

In [2]:
import sys
from pathlib import Path

for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / 'dataset').exists() and (candidate / 'paths.py').exists():
        candidate_str = str(candidate.resolve())
        if candidate_str not in sys.path:
            sys.path.insert(0, candidate_str)
        break

## 0. Prerequisites

Needs the same local services as the simulation step: FHIR (`04_start_fhir.sh`) and Qdrant (`05_start_qdrant.sh`),
plus a diagnosis dataset already built (step 3). `OPENAI_API_KEY` is needed for the `Plan` tool and for
`ProcedureSearch`'s embedding call (loads `jinaai/jina-embeddings-v3` locally, no API key needed for the
embedding itself, but downloads a ~1GB model the first time).

In [15]:
import requests

fhir_ok = requests.get("http://localhost:8080/fhir/metadata", timeout=5).status_code == 200
qdrant_ok = requests.get("http://localhost:6333/collections", timeout=5).status_code == 200
print("FHIR reachable:", fhir_ok)
print("Qdrant reachable:", qdrant_ok)

FHIR reachable: True
Qdrant reachable: True


In [4]:
import sys

if "ipykernel" in sys.modules:
    import nest_asyncio
    nest_asyncio.apply()  # lets us `await` the async tool_execs functions directly in notebook cells

## 1. Load a patient (input side)

Pick a diagnosis dataset (built in step 3) and one admission out of it — this becomes `patient_data`, the same
object `prepare_patient()` in `src/runs/run.py` passes to every tool.

In [5]:
from paths import DIAGNOSIS_DATASETS_DIR
from dataset.mimic_dataset import MIMIC_Dataset

available_datasets = sorted(p.name for p in DIAGNOSIS_DATASETS_DIR.glob("*") if p.is_dir())
print("available datasets:", available_datasets)

DATASET_NAME = available_datasets[0]  # e.g. "appendicitis"
ds = MIMIC_Dataset.load_dataset(DATASET_NAME)

hadm_id = next(iter(ds.hadm_ids))
patient_data = ds[hadm_id]
print("Using diagnosis:", DATASET_NAME, "| hadm_id:", hadm_id)

available datasets: ['appendicitis']


15it [00:00, 199.52it/s]

Using diagnosis: appendicitis | hadm_id: 20345216


## 2. Set up FHIR context (organization, practitioner, patient)

Mirrors the first few lines of `prepare_patient()` in `src/runs/run.py`: every tool call needs an
`organization_id`, `practitioner_id`, `patient_id`, a `requests.Session`, and `headersList` to talk to the local
FHIR server.

In [6]:
from backend.fhir_client import create_fhir_session, headersList, base_url, post_fhir_resource
from backend.fhir_setup import setup_org_and_practitioner, generate_patient_resource

session = create_fhir_session()

organization_id, practitioner_id = setup_org_and_practitioner(
    base_url=base_url, headers_list=headersList, session=session
)

patient = generate_patient_resource(patient_data.patients, practitioner_id)
patient_id = post_fhir_resource(patient, headersList, session=session)

print("organization_id:", organization_id)
print("practitioner_id:", practitioner_id)
print("patient_id:", patient_id)

organization_id: 102
practitioner_id: 103
patient_id: 104


In [ ]:
# Common kwargs every tool exec function needs on top of its own (LLM-supplied) arguments
# this is what MedAssistant.execute_tool_calls() normally assembles automatically via PatientContext.to_dict()
common_ctx = dict(
    patient_id=patient_id,
    patient_hadm_id=hadm_id,
    organization_id=organization_id,
    practitioner_id=practitioner_id,
    session=session,
    headersList=headersList,
    patient_data=patient_data,
)

## 3. Exercise each tool: input → tool → output

Each block below shows the **input** an LLM tool-call would send, calls the matching **exec function** (the
tool's actual implementation), and prints the **output** string that would be fed back into the conversation.

### 1. `PhysicalExamination` → `get_physical_exam_results`
**Input:** none (empty tool schema). **Under the hood:** reads the LLM-extracted PE text from the built dataset
(`patient_data.history_pe_admedication_diagnosis["pe"]`); posts `ServiceRequest`+`Observation` to FHIR.

In [8]:
from tool_execs import get_physical_exam_results

tool_input = {}  # PhysicalExamination() has no fields
output = await get_physical_exam_results(**common_ctx, **tool_input)
print(output)

Physical Examination:
   On admission:  Vitals: 98.1 95 124/61 22 96%   GEN: A&O, NAD HEENT: No scleral icterus, mucus membranes moist CV: RRR, No M/G/R PULM: Clear to auscultation b/l, No W/R/R ABD: Soft, very TTP RLQ, +obturator sign, -Rosving sign. No rebound, some voluntary gaurding Ext: No ___ edema, ___ warm and well perfused


### 2. `LabRequestList` → `get_blood_value_results`
**Input:** one or more `BloodValue` enum members (MIMIC lab item names).
**Under the hood:** filters `patient_data.lab_events` (fluid == Blood) for the earliest matching result in the
first 24h; posts `ServiceRequest`+`Observation` to FHIR.

In [9]:
from tool_execs import get_blood_value_results
from tools import BloodValue

tool_input = {"lab_values": [{"lab_value": BloodValue._50803}]}  # Calculated Bicarbonate, Whole Blood
output = await get_blood_value_results(**common_ctx, **tool_input)
print(output)

Calculated Bicarbonate, Whole Blood: N/A



### 3. `UrineRequestList` → `get_urine_value_results`
**Input:** one or more `UrineValue` enum members. **Under the hood:** same as labs but `fluid == "Urine"`.

In [10]:
from tool_execs import get_urine_value_results
from tools import UrineValue

tool_input = {"urine_values": [{"urine_value": UrineValue._51486}]}  # Leukocytes
output = await get_urine_value_results(**common_ctx, **tool_input)
print(output)

Leukocytes: N/A



### 4. `MicrobiologyRequestList` → `get_microbiology_results`
**Input:** one or more `MicroBiologyValue` enum members. **Under the hood:** filters `patient_data.microbiology`,
groups by specimen, generates up to 3 chained Observations (test/organism/susceptibility).

In [11]:
from tool_execs import get_microbiology_results
from tools import MicroBiologyValue

tool_input = {"microbiology_tests": [{"microbiology_value": MicroBiologyValue._90144}]}  # TOXOPLASMA IgG ANTIBODY
output = await get_microbiology_results(**common_ctx, **tool_input)
print(output)

TOXOPLASMA IgG ANTIBODY: N/A



### 5. `RadiologyRequestFHIR` → `get_radiology_results`
**Input:** `modality` + `region` enums, optional free-text `info`.
**Under the hood:** filters `patient_data.radiology` by modality/region; posts `ServiceRequest`+`DiagnosticReport`.

In [12]:
from tool_execs import get_radiology_results
from tools import RadiologyModalityValue, RadiologyRegionValue

tool_input = {
    "modality": RadiologyModalityValue.CT,
    "region": RadiologyRegionValue.Abdomen,
    "info": "Evaluate for suspected diagnosis.",
}
output = await get_radiology_results(**common_ctx, **tool_input)
print(output)

Radiology Report (CT
Abdomen):

Radiology Report:
    Examination could not be performed.


### 6. `MedicationRequestList` → `get_medication_results`
**Input:** one or more medication orders (drug name, dosage, route, frequency).
**Under the hood:** *no ground-truth lookup* — this is a simulated confirmation (echoes the request back with a
timestamp) plus NDC/RxNorm/SNOMED/ATC code resolution for the drug name; posts `MedicationRequest`+`Observation`.

In [13]:
from tool_execs import get_medication_results

tool_input = {
    "medications": [
        {
            "drug_name": "Amoxicillin",
            "dosage_text": "500mg Tablet",
            "dosage_value": 500,
            "dosage_unit": "mg",
            "period": 1,
            "period_unit": "d",
            "frequency": 3,
            "route": "Oral",
        }
    ]
}
output = await get_medication_results(**common_ctx, **tool_input)
print(output)

Amoxicillin: 500mg Tablet 500 mg, 3 x 1d, Oral



### 7. `ProcedureSearch` → `get_procedure_search_results`
**Input:** short free-text procedure description.
**Under the hood:** semantic search against the Qdrant `mimic_iv_icd_codes_procedures` collection (built in
`build_procedure_db.ipynb`), embedding the query locally with `jinaai/jina-embeddings-v3` (downloads ~1GB the
first time this runs). Returns up to 10 candidate procedure titles.

In [14]:
from tool_execs import get_procedure_search_results

tool_input = {"procedure": "appendectomy"}  # swap for a procedure relevant to your dataset's diagnosis
output = await get_procedure_search_results(**common_ctx, **tool_input)
print(output)

flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn i

collections=[CollectionDescription(name='mimic_iv_icd_codes_procedures')]
points=[ScoredPoint(id='2bfc35aa-e5bf-4c0c-a3cd-2c0fab62911b', version=379, score=0.9536556, payload={'icd_code': '4709', 'icd_version': 9, 'long_title': 'Other appendectomy'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id='e3f1f664-d881-4c03-97a7-de62503b5d2f', version=812, score=0.9536556, payload={'icd_code': '4709', 'icd_version': 9, 'long_title': 'Other appendectomy'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id='e77be921-7c5b-40c8-a528-35bb6dddf09e', version=379, score=0.9083955, payload={'icd_code': '4791', 'icd_version': 9, 'long_title': 'Appendicostomy'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id='b9436acb-dc6e-4e2a-abfd-6da9137f217b', version=812, score=0.9083955, payload={'icd_code': '4791', 'icd_version': 9, 'long_title': 'Appendicostomy'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id='151284ab-893a-4033-b668-afd7cad3caa6', ver

### 8. `ProcedureRequestFHIR` → `get_procedure_request_results`
**Input:** an exact procedure name (ideally one returned by `ProcedureSearch` above).
**Under the hood:** exact match against the local ICD procedure catalog, else Qdrant fallback; posts
`ServiceRequest`+`Procedure`.

In [15]:
from tool_execs import get_procedure_request_results

tool_input = {"procedure": "Laparoscopic appendectomy"}  # use a title from the ProcedureSearch output above
output = await get_procedure_request_results(**common_ctx, **tool_input)
print(output)

Procedure service request: resource_type='ServiceRequest' fhir_comments=None id=None implicitRules=None implicitRules__ext=None language=None language__ext=None meta=None contained=None extension=None modifierExtension=None text=None asNeededBoolean=None asNeededBoolean__ext=None asNeededCodeableConcept=None authoredOn=datetime.datetime(2026, 7, 23, 21, 42, 40, 230826) authoredOn__ext=None basedOn=None bodySite=None bodyStructure=None category=None code=CodeableReference(resource_type='CodeableReference', fhir_comments=None, extension=None, id=None, concept=CodeableConcept(resource_type='CodeableConcept', fhir_comments=None, extension=None, id=None, coding=None, text='Laparoscopic appendectomy', text__ext=None), reference=None) doNotPerform=None doNotPerform__ext=None encounter=None focus=None identifier=None instantiatesCanonical=None instantiatesCanonical__ext=None instantiatesUri=None instantiatesUri__ext=None insurance=None intent='order' intent__ext=None location=None note=None oc

### 9. `Finish` → `finish`
**Input:** free-text `diagnosis`. **Under the hood:** pure local function, no FHIR/Qdrant/dataset access — just
returns the diagnosis string. This is also what ends the simulation loop (`assistants.py` checks for the
function name `"finish"`).

In [16]:
from tool_execs import finish

tool_input = {"diagnosis": "Uncomplicated acute appendicitis"}
output = finish(**tool_input)
print(output)

Uncomplicated acute appendicitis


### 10. `Plan` → `generate_routine`
**Input:** none from the LLM directly — internally builds a prompt from the tool schemas and the conversation
so far. **Under the hood:** calls OpenAI's reasoning model (`o1`, see `config.REASONING_MODEL`) directly — no
FHIR/Qdrant/dataset access. Requires `OPENAI_API_KEY`.

In [20]:
  from dotenv import load_dotenv
  from paths import SRC_DIR
  load_dotenv(SRC_DIR / ".env", override=True)

True

In [21]:
from tool_execs import generate_routine

tools_for_planning_routine = []  # normally the other 9 tool schemas minus Plan itself; empty is fine for a smoke test
patient_info = "65yo male with RLQ abdominal pain, fever, and elevated WBC."
output = generate_routine(tools=tools_for_planning_routine, patient_info=patient_info)
print(output)

1. Assess Completeness of Current Data  
   - Verify that we have:  
     • Detailed pain history (onset, location specifics, duration, nature)  
     • Current vital signs (temperature, heart rate, blood pressure, respiratory rate)  
     • Physical examination report (particularly abdominal exam, palpation findings, rebound tenderness, Rovsing’s sign, etc.)  
     • Existing lab results (WBC differential, any inflammatory markers)  
   - If any of these are missing or incomplete, gather or repeat them.

2. Order Comprehensive Laboratory Tests  
   - LabValueRequestList:  
     • Complete Blood Count (CBC) with differential (including neutrophils, lymphocytes, monocytes)  
     • C-Reactive Protein (CRP)  
     • Erythrocyte Sedimentation Rate (ESR)  
     • Basic Metabolic Panel (Sodium, Potassium, Chloride, Bicarbonate, Blood Urea Nitrogen, Creatinine, Glucose, eGFR)  
     • Liver Function Tests (Aspartate Aminotransferase [AST], Alanine Aminotransferase [ALT], Alkaline Phosphatase

### 11. `PatientHistory` → `patient_history` (pancreatic_cancer dataset only)
**Input:** none — keyed internally off `patient_hadm_id`. **Under the hood:** reads
`src/resources/pancreatic_cancer_info.json` (built by `extract_pancreatic_cancer_info.ipynb`); no FHIR/Qdrant.
Only registered as a tool when `ds_name == "pancreatic_cancer"` (`src/runs/run.py`).

In [22]:
from tool_execs import patient_history
from paths import PANCREATIC_CANCER_INFO_PATH

if DATASET_NAME == "pancreatic_cancer" and PANCREATIC_CANCER_INFO_PATH.exists():
    output = patient_history(patient_hadm_id=hadm_id)
    print(output)
else:
    print(f"Skipping — dataset is {DATASET_NAME!r}, PatientHistory is only wired up for 'pancreatic_cancer'.")

Skipping — dataset is 'appendicitis', PatientHistory is only wired up for 'pancreatic_cancer'.


## 4. Bonus: tools used only in the bias / optional-admission run variants

Not part of the baseline (`run.py`) tool set, but registered in `run_with_sex_bias.py` / `run_optional_admission.py`.

In [23]:
from tool_execs import get_vitalsign_results

# VitalSigns — reads patient_data.triage (temperature/heartrate/resprate/o2sat/sbp/dbp), posts an Observation.
output = await get_vitalsign_results(**common_ctx)
print(output)

Vital Signs:
    Body temperature: 98.1 Fahrenheit
    Heart rate: 95 /min
    Respiratory rate: 22 /min
    Oxygen saturation by Pulse oximetry: 96 %
    Systolic blood pressure: 124 mmHg
    Diastolic blood pressure: 61 mmHg


In [24]:
from tool_execs import close_case

# CloseCase — replaces Finish in the bias/optional-admission variants; pure local function, no I/O.
tool_input = {
    "diagnosis": "Uncomplicated acute appendicitis",
    "decision": "admission",
    "reasoning": "Peritoneal signs on exam, needs surgical management.",
}
output = close_case(**tool_input)
print(output)

{'diagnosis': 'Uncomplicated acute appendicitis', 'decision': 'admission', 'reasoning': 'Peritoneal signs on exam, needs surgical management.'}


In [ ]:
import os
print("cached OPENAI_API_KEY repr:", repr(os.environ.get("OPENAI_API_KEY")))


In [26]:
from dotenv import load_dotenv
from paths import SRC_DIR
load_dotenv(SRC_DIR / ".env", override=True)
import os
print("key loaded:", bool(os.environ.get("OPENAI_API_KEY")), "len:", len(os.environ.get("OPENAI_API_KEY", "")))


key loaded: True len: 164
